# Weak decays

:::{autolink-concat}
:::

The decay $\Lambda_c^+ \to \Lambda K^+ \pi^0$ is a charm decay, so the node where the $\Lambda_c^+$ splits into a resonance and a spectator particle is mediated by the weak interaction. The resonance itself, however, decays strongly. A decay chain like this mixes interaction types within one topology, which means that the interaction type has to be chosen for each node separately. This notebook shows how to do that with the {class}`.StateTransitionManager` and how to inspect the resulting resonances per subsystem.

## Unconstrained nodes

By default, {mod}`qrules` allows the {attr}`~.InteractionType.STRONG`, {attr}`~.InteractionType.EM`, and {attr}`~.InteractionType.WEAK` interaction on every node. Since $\Lambda_c^+$ is a charmed baryon, the weak interaction has to be available somewhere in the decay chain, so it cannot simply be switched off globally:

:::{margin}

The `mass_conservation_factor` is the number of widths by which the outgoing states may exceed the available mass in {class}`.MassConservation`. The default of $3$ is permissive, while $0.6$ keeps the resonances within the phase space that $\Lambda_c^+$ actually offers.

:::

In [ ]:
import qrules

unconstrained = qrules.generate_transitions(
    initial_state="Lambda(c)+",
    final_state=["Lambda", "K+", "pi0"],
    mass_conservation_factor=0.6,
)
print("found", len(unconstrained.transitions), "solutions")
sorted(unconstrained.get_intermediate_particles().names)

Most of these candidates are not physical. Take $\Delta(1232)^0$: it carries strangeness $0$, so its decay into $\Lambda\pi^0$ changes strangeness and can only proceed weakly. Solutions like this one contain two weak nodes and are therefore heavily suppressed (see the interaction strengths in [](reaction.ipynb)).

## One weak node

The isobar topology for a three-body decay has two nodes: node $0$ is the production node, where the $\Lambda_c^+$ decays into a resonance and a spectator particle, and node $1$ is the node where the resonance decays into the two remaining final state particles. It is that second node that we restrict to the strong interaction, while node $0$ keeps its default settings and can be weak:

In [ ]:
from qrules import InteractionType, StateTransitionManager

stm = StateTransitionManager(
    initial_state=["Lambda(c)+"],
    final_state=["Lambda", "K+", "pi0"],
    mass_conservation_factor=0.6,
)
stm.set_allowed_interaction_types([InteractionType.STRONG], node_id=1)
problem_sets = stm.create_problem_sets()
reaction = stm.find_solutions(problem_sets)

print("found", len(reaction.transitions), "solutions")
sorted(reaction.get_intermediate_particles().names)

In [ ]:
n_resonances = len(reaction.get_intermediate_particles())
assert n_resonances == 10, n_resonances

Ten resonances survive, each of which conserves strangeness in its own decay. The {attr}`~.ReactionInfo.transitions` are still the ones that contain a weak production node, but the resonances are now genuine strong isobars.

## Resonances per subsystem

The three final state particles form three two-body subsystems, and each resonance belongs to exactly one of them. Which one can be read off from the topology: the resonance sits on the single intermediate edge, and {meth}`~.Topology.get_originating_final_state_edge_ids` tells us which final state edges come out of the node that it decays into.

In [ ]:
from collections import defaultdict


def group_by_subsystem(transitions):
    groups = defaultdict(list)
    for transition in transitions:
        topology = transition.topology
        (resonance_id,) = topology.intermediate_edge_ids
        decay_node = topology.edges[resonance_id].ending_node_id
        final_state_ids = topology.get_originating_final_state_edge_ids(decay_node)
        subsystem = tuple(
            sorted(transition.states[i].particle.latex for i in final_state_ids)
        )
        groups[subsystem].append(transition)
    return groups


subsystems = group_by_subsystem(reaction.transitions)
{subsystem: len(transitions) for subsystem, transitions in subsystems.items()}

Each group can be rendered as a Mermaid flowchart with {func}`.asmermaid`. Collapsing by `"topology"` collects all resonances that {mod}`qrules` found on the intermediate edge into a single node:

In [ ]:
from IPython.display import Markdown, display

from qrules import io

for (first, second), transitions in subsystems.items():
    diagram = io.asmermaid(transitions, collapse="topology", markdown=True)
    display(Markdown(f"${first}{second}$ subsystem:\n\n{diagram}"))

The $K^+\pi^0$ subsystem is limited to the two lightest kaon resonances, because the $\Lambda$ spectator leaves less than $1.2$ GeV for the $K^+\pi^0$ pair. The other two subsystems have more phase space available and therefore show a richer spectrum of $N^{*+}$ and $\Sigma^{*0}$ states.

:::{seealso}

[](reaction.ipynb) and [](visualize.ipynb)

:::